# 用講的跟 Workflow 對話

語音牽動兩個模組，這一份兩個都示範：

- **Action** — `VoiceAnswerAction` 同時產生兩個頻道：說出口的和顯示在畫面上的。
- **Perceive** — `VoiceTextPerceive` 把說出來的話變成這一輪的輸入，並在有人開口時中止正在進行的回答。

四段依序回答四個問題：打字問可以用聲音答嗎、用講的怎麼問、聲音和文字怎麼同時出去、講到一半被打斷會怎樣。

**前四段用假的模型與假的音訊，不需要網路也不需要憑證**，在 Colab 直接跑得動。最後一段才接真的端點。


## 在 Colab 準備環境


In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 一、打字問，用聲音答（只動 Action）

語音輸出不需要語音輸入。這一段的 perceive 是最普通的 `PassThroughPerceive`，使用者用打的，Agent 用講的。

`VoiceAnswerAction` 一次產生兩個頻道：`spoken` 是口語、講判斷與理由；`displayed` 是要用看的——型號、數字、條列。**兩者互相補充，不是把畫面唸一遍。**


In [ ]:
import json
from agentic_sdk import Workflow
from agentic_sdk.audio import FakeAudioOutput
from agentic_sdk.modules import PassThroughPerceive, VoiceAnswerAction

from types import SimpleNamespace


class FakeModel:
    """一個逐字吐出答案的假模型，這樣不用憑證也能看到串流的行為。
    正式使用時把它換成真的端點設定即可。"""

    def __init__(self, answer):
        self._answer = answer
        self.produced = 0
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, **_kwargs):
        def chunks():
            for character in self._answer:
                self.produced += 1          # 目前為止吐了幾個字
                yield SimpleNamespace(
                    choices=[SimpleNamespace(delta=SimpleNamespace(content=character, tool_calls=None),
                                             finish_reason=None)],
                    model="fake", usage=None)
        return chunks()

speaker = FakeAudioOutput()
action = VoiceAnswerAction(speech=speaker, api_key="k", base_url="https://example.test/v1", model="m")
action._client = FakeModel(json.dumps(
    {"spoken": "保固十二個月，延長保固可以再加兩年。",
     "displayed": "保固期 12 個月；延長保固再加 24 個月；配件不在範圍內"}, ensure_ascii=False))

workflow = Workflow(workflow_name="打字問、聲音答",
                    perceive=PassThroughPerceive(), action=action)
result = workflow.run("保固多久？")

print("畫面上顯示的 :", result.final_message.replace("\n", " / "))
print("說出口的     :", speaker.spoken[0])

## 二、用講的問（加上 Perceive）

把 `PassThroughPerceive` 換成 `VoiceTextPerceive`，其餘不變。

兩件事值得注意。**安靜的片段不會離開這台機器**——純靜音和說話計費相同，而且會被服務辨識成沒有人說過的字。還有，**話比 `run()` 先到**：語音什麼時候來取決於人什麼時候想講，所以模組先收著，`run()` 不必再被告知一次。


In [ ]:
import struct
from agentic_sdk.audio import FakeAudioInput
from agentic_sdk.modules import VoiceTextPerceive

def silence(frames=1600):
    return struct.pack(f"<{frames}h", *([0] * frames))

def speech(frames=1600, level=8000):
    return struct.pack(f"<{frames}h", *([level, -level] * (frames // 2)))

microphone = FakeAudioInput()
perceive = VoiceTextPerceive(transport=microphone)

for chunk in [silence(), speech(), speech(), silence()]:
    perceive.hear(chunk)
print("送給服務的片段數 :", len(microphone.sent), "（前面的靜音沒有離開這台機器）")

microphone.transcribe("保固多久？")
print("開跑前就拿到的話 :", perceive.pending_input())

speaker = FakeAudioOutput()
action = VoiceAnswerAction(speech=speaker, api_key="k", base_url="https://example.test/v1", model="m")
action._client = FakeModel(json.dumps(
    {"spoken": "保固十二個月。", "displayed": "保固期 12 個月"}, ensure_ascii=False))

talking = Workflow(workflow_name="用講的問、用聲音答", perceive=perceive, action=action)
result = talking.run()
print("回答         :", result.final_message)
print("說出口的     :", speaker.spoken[0])

## 三、聲音還在講，畫面還在跑字

`spoken` 這個欄位一寫完就送去合成，**不等整段回覆結束**。回覆愈長，這個提早開口愈有感。

下面用一個會計數的假模型，在開始說話的那一刻記下它總共才吐了幾個字。


In [ ]:
started_speaking_at = []

class WatchingSpeaker(FakeAudioOutput):
    """在開始說話的那一刻，記下模型總共才吐了幾個字。"""
    def speak(self, text):
        started_speaking_at.append(model.produced)
        return super().speak(text)

answer = json.dumps({"spoken": "保固十二個月，另外配件另計。",
                     "displayed": "保固期 12 個月；配件不在範圍內；延長保固可再加 24 個月"},
                    ensure_ascii=False)

model = FakeModel(answer)
speaker = WatchingSpeaker()
action = VoiceAnswerAction(speech=speaker, api_key="k", base_url="https://example.test/v1", model="m")
action._client = model

workflow = Workflow(workflow_name="邊說邊顯示", perceive=PassThroughPerceive(), action=action)
stream = workflow.stream("保固多久？")
for _piece in stream:
    pass

print("整段回覆長度         :", len(answer), "個字")
print("開始說話時模型才吐了 :", started_speaking_at[0], "個字")
print("→ spoken 一寫完就送去合成，displayed 還在後面繼續寫")
print("說出口的             :", speaker.spoken[0])
print("畫面上最後顯示的     :", stream.result.final_message)

## 四、講到一半被打斷

偵測靠的是**語音活動，不是聽懂了什麼**——開口約 600 毫秒就測得到，轉寫要將近四秒，等字就等於繼續講在別人身上。

被打斷之後，記憶留下的是**對方實際收到的那一段**，不是模型寫完的整段。沒收到的部分不會進入下一輪的上下文，否則 Agent 會引用一句沒人聽過的話。

注意 `aborted` 是 `False`：那個旗標是流程自我中止（跳轉上限、逾時）才用的，會被當成錯誤顯示。**有人故意插話不是錯誤。**


In [ ]:
from agentic_sdk.audio import heard_portion
from agentic_sdk.core import ModuleOutput
from agentic_sdk.core.cancellation import CancellationToken

class SlowAnswer:
    """一個講很久的 action，讓我們有時間插話。"""
    name = "action"
    def __init__(self, microphone, speaker):
        self._microphone, self._speaker = microphone, speaker
    def __call__(self, state):
        text = "保固期是十二個月，延長保固可以再加兩年，另外配件另計"
        for index, _piece in enumerate(self._speaker.speak(text)):
            if index == 1:
                self._microphone.start_speaking()          # 使用者在這裡開口
            if state.should_stop():
                state.report_delivered(heard_portion(text, 2.0))
                state.cancel.raise_if_cancelled()
        state.report_delivered(text)
        return ModuleOutput(next_module=None, payload={"latest_final_message": text})

microphone = FakeAudioInput()
listening = VoiceTextPerceive(transport=microphone)
talking = Workflow(workflow_name="會被打斷的對話", perceive=listening,
                   action=SlowAnswer(microphone, FakeAudioOutput()))
microphone.transcribe("保固多久？")
cut = talking.run(cancel=CancellationToken())

print("被打斷了嗎 :", cut.interrupted)
print("這是錯誤嗎 :", cut.aborted)
print("對方收到的 :", cut.interrupt_payload["delivered"])
print("記憶裡留的 :", talking.memory.turns[-1].content)

## 五、接上真的端點

端點設定的形狀前面幾份教材已經看過了，這裡只有一個地方不一樣：**音訊來源是一個物件，不是三個設定。**

聊天端點是同構的，三件式描述得完。音訊來源不是——即時轉寫是長連線、合成是串流回應，各自的握手與生命週期都不同——所以它們在外面建好再交進去。`turn_detection` 也在這裡：**服務怎麼判斷你講完了，是傳輸的事，不是模組的事。**


In [ ]:
from agentic_sdk.audio.realtime import RealtimeTranscription
from agentic_sdk.audio.speech import SpeechOutput

listening = RealtimeTranscription(
    api_key="<OPENAI_API_KEY>",
    model="gpt-4o-transcribe",
    language="zh",
    # 靠模型判斷句末；low 會等久一點，讓換氣或講得慢的人把話講完。
    turn_detection={"type": "semantic_vad", "eagerness": "low"},
)
speaking = SpeechOutput(api_key="<OPENAI_API_KEY>", model="gpt-4o-mini-tts", voice="alloy")

perceive = VoiceTextPerceive(transport=listening, speech_threshold=500, hangover_seconds=1.2)
action = VoiceAnswerAction(speech=speaking, api_key="<OPENAI_API_KEY>", model="gpt-5")

## 六、端點不是 OpenAI 時

上面是 SDK 唯一知道的形狀：**它只會建 `OpenAI` client，不內建任何其他廠商。**

你的端點如果連線方式不同——不同的位址規則、不同的認證、額外的查詢參數——**覆蓋一個方法就好**。開連線之後的一切（session 設定、送音訊、事件分派、回合判定、靜音閘門）全部繼承。


In [ ]:
from openai import AzureOpenAI          # 換成你那一家的 client
from agentic_sdk.audio.realtime import RealtimeTranscription


class MyTranscription(RealtimeTranscription):
    """我自己接的端點。SDK 不知道也不負責這一家。"""

    def _open(self):
        return AzureOpenAI(
            azure_endpoint="https://<資源>.cognitiveservices.azure.com",
            api_key="<KEY>",
            api_version="2025-04-01-preview",
        ).beta.realtime.connect(
            model=self._model,
            extra_query={"intent": "transcription"},
        )


# 底下完全不變——模組分不出這是你寫的還是 SDK 附的
perceive = VoiceTextPerceive(transport=MyTranscription(model="<部署名稱>", language="zh"))

## SDK 不擷取麥克風，也不播放聲音

進來的音訊由你餵給 `hear()`；出去的要包一層在 `speak()` 裡交給音訊裝置再 `yield`——**先播放再 `yield`**，插話時放棄串流才會同時停掉播放與合成。

可執行的完整範例在 repo 的 `examples/voice/desktop_voice_agent.py`，用一個 WAV 當麥克風，沒有音效裝置也跑得動。
